# 📊 Notebook 2 — Visualisation & Analyses Exploratoires V12

Plots sur `dataset_final_v12_couche1.csv` (8 400 × 741, 62 cibles, 240 pays, 8 clusters).

## Sections
1. Couverture des 62 cibles
2. **Cultures spécifiques** — rendements par culture
3. **Profil des 8 clusters climatiques**
4. **EcoCrop suitability scores** — visualisation par culture/cluster
5. **Religion Pew 2010** — distributions par pays
6. **Meat consumption per type** — habitudes alimentaires
7. **Stress climatique** (heat/frost/aridity/PET)
8. Corrélations features → cibles (par groupe)
9. Évolution temporelle 4 cibles × 8 pays
10. Plafond de prédictibilité par cible

## 1. Imports & chargement V12

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 90

# Charge V12 (Couche 1) avec fallback
for p in ['data/cleaned/dataset_final_v12_couche1.csv',
          'data/cleaned/dataset_final_v11_couche1.csv',
          'data/cleaned/dataset_final_v10_couche1.csv',
          'data/cleaned/dataset_final_v9_couche1.csv',
          'data/cleaned/dataset_final_v8_honest.csv']:
    if os.path.exists(p):
        df = pd.read_csv(p, low_memory=False)
        print(f'✓ Chargé : {p}')
        break

df = df.dropna(subset=['ISO']).copy()
df['ISO'] = df['ISO'].astype(str)
df['Annee'] = df['Annee'].astype(int)
if 'cluster' in df.columns: df['cluster'] = df['cluster'].astype(int)

targets = [c for c in df.columns if c.startswith('target_')]
print(f'Dataset: {df.shape}, {df["ISO"].nunique()} pays, {len(targets)} cibles')

## 2. Couverture des 62 cibles

In [ ]:
target_coverage = pd.Series({t: df[t].notna().sum() for t in targets}).sort_values()
fig, ax = plt.subplots(figsize=(11, max(8, len(targets)*0.18)))
colors = ['#d62728' if v < 1000 else ('#ff7f0e' if v < 3000 else '#2ca02c') for v in target_coverage.values]
target_coverage.plot.barh(color=colors, ax=ax)
ax.set_title(f"Couverture des {len(targets)} cibles V12 — observations non-nulles", weight='bold', fontsize=13)
ax.set_xlabel('Nombre de lignes (sur 8 400)')
ax.axvline(1000, color='red', alpha=0.3, ls='--', label='<1000 : à éviter')
ax.axvline(3000, color='orange', alpha=0.3, ls='--', label='<3000 : limité')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Cultures spécifiques — Distributions des rendements

**Justification** : on remplace les agrégats faibles (R²=0.09-0.14) par modèles dédiés par culture. Voici les distributions log(rendement) pour les 18 cultures les plus couvertes.

In [ ]:
specific_yields = [c for c in df.columns if c.startswith('yield_')
                   and not c.endswith('_kgha') and df[c].notna().sum() > 1500]
specific_yields = sorted(specific_yields, key=lambda c: -df[c].notna().sum())[:18]

fig, axes = plt.subplots(3, 6, figsize=(20, 11))
for ax, c in zip(axes.flatten(), specific_yields):
    s = df[c].dropna()
    ax.hist(np.log1p(s), bins=30, color='forestgreen', alpha=0.7, edgecolor='white')
    ax.axvline(np.log1p(s.median()), color='red', ls='--', lw=1.5, label=f'med={s.median():.0f}')
    ax.set_title(f'{c.replace("yield_","")}\nn={len(s):,}', fontsize=10, weight='bold')
    ax.set_xlabel('log1p(kg/ha)', fontsize=8)
    ax.legend(fontsize=7)
plt.suptitle('Distributions log-rendement de 18 cultures spécifiques', weight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Profil des 8 clusters climatiques

In [ ]:
cluster_profile = pd.read_csv('data/cleaned/country_clusters.csv')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plot 1 : T° × P, coloré par cluster
ax = axes[0]
for c in sorted(cluster_profile['cluster'].unique()):
    sub = cluster_profile[cluster_profile['cluster']==c]
    ax.scatter(sub['temp_mean'], sub['precip_mean'], label=f'C{c} (n={len(sub)})',
               s=60, alpha=0.7, edgecolor='black', lw=0.5, color=plt.cm.tab10(c))
ax.set_xlabel('Température moyenne (°C)')
ax.set_ylabel('Précipitations annuelles (mm)')
ax.set_title('Pays dans l\'espace climatique', weight='bold')
ax.legend(loc='best', fontsize=9)

# Plot 2 : Latitude × T°
ax = axes[1]
for c in sorted(cluster_profile['cluster'].unique()):
    sub = cluster_profile[cluster_profile['cluster']==c]
    ax.scatter(sub['latitude'], sub['temp_mean'], label=f'C{c}',
               s=60, alpha=0.7, edgecolor='black', lw=0.5, color=plt.cm.tab10(c))
ax.set_xlabel('Latitude (°)')
ax.set_ylabel('Température (°C)')
ax.set_title('Latitude vs Température par cluster', weight='bold')
ax.axvline(0, color='gray', ls=':', alpha=0.5)
ax.legend(fontsize=9)

# Plot 3 : Taille des clusters
ax = axes[2]
counts = cluster_profile['cluster'].value_counts().sort_index()
ax.bar(counts.index, counts.values, color=[plt.cm.tab10(c) for c in counts.index], alpha=0.85)
ax.set_xlabel('Cluster')
ax.set_ylabel('Nombre de pays')
ax.set_title('Taille de chaque cluster', weight='bold')
for i, v in enumerate(counts.values):
    ax.text(counts.index[i], v + 1, str(v), ha='center', fontsize=10, weight='bold')

plt.tight_layout()
plt.show()

## 5. 🆕 EcoCrop Suitability Scores — par culture

**Justification d'usage** : On utilise la base FAO EcoCrop (T_opt, P_opt, frost tolerance) pour calculer un score de viabilité climatique de chaque culture dans chaque pays. Score = 0 (impossible) à 1 (idéal).

In [ ]:
# Heatmap suitability × cluster pour 16 cultures clés
suit_cols = [c for c in df.columns if c.startswith('suit_')]
if suit_cols:
    KEY_CROPS = ['suit_wheat','suit_rice','suit_maize','suit_soybeans','suit_potato',
                 'suit_tomato','suit_apple','suit_banana','suit_coconut','suit_olives',
                 'suit_grape','suit_dates','suit_strawberry','suit_orange','suit_mango','suit_avocado']
    KEY_CROPS = [c for c in KEY_CROPS if c in df.columns]

    # Moyenne par cluster
    pivot = df.groupby('cluster')[KEY_CROPS].mean().T
    pivot.index = [c.replace('suit_','') for c in pivot.index]

    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(pivot, cmap='RdYlGn', vmin=0, vmax=1, annot=True, fmt='.2f',
                cbar_kws={'label': 'Suitability score (0-1)'}, ax=ax)
    ax.set_title('Suitability EcoCrop par culture × cluster climatique', weight='bold', fontsize=13)
    ax.set_xlabel('Cluster')
    ax.set_ylabel('Culture')
    plt.tight_layout()
    plt.show()
    print('\n→ Vert = idéal, Rouge = impossible. Note : datte uniquement viable en zone aride chaude (cluster 1).')

In [ ]:
# Validation : suitability tomato vs rendement réel tomato (corrélation)
if 'suit_tomato' in df.columns and 'yield_tomato' in df.columns:
    sub = df.dropna(subset=['suit_tomato','yield_tomato'])
    r = sub[['suit_tomato','yield_tomato']].corr().iloc[0,1]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(sub['suit_tomato'], np.log1p(sub['yield_tomato']),
               c=sub['cluster'], cmap='tab10', alpha=0.4, s=15)
    ax.set_xlabel('Suitability EcoCrop tomate')
    ax.set_ylabel('log1p(yield_tomato_kgha)')
    ax.set_title(f'Validation suitability tomate vs rendement réel\nr={r:+.3f}',
                  weight='bold', fontsize=12)
    plt.tight_layout()
    plt.show()

## 6. 🆕 Religion Pew 2010 — distribution par pays

**Justification** : religion influence régime alimentaire (porc, alcool, bovins sacrés) → utile pour prédire élevage.

In [ ]:
rel_cols = [c for c in df.columns if c.startswith('share_') and any(r in c for r in ['muslim','christian','hindu','buddhist','unaffiliated','folk','jew'])]
if rel_cols:
    # Distribution par cluster
    rel_by_cluster = df.groupby('cluster')[rel_cols].mean()
    rel_by_cluster.columns = [c.replace('share_','').replace('_pct','') for c in rel_by_cluster.columns]

    fig, ax = plt.subplots(figsize=(13, 7))
    rel_by_cluster.plot(kind='bar', stacked=True, ax=ax, colormap='tab10', alpha=0.85)
    ax.set_title('Composition religieuse moyenne par cluster climatique', weight='bold', fontsize=13)
    ax.set_xlabel('Cluster')
    ax.set_ylabel('% population')
    ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=10)
    ax.set_xticklabels([f'C{c}' for c in rel_by_cluster.index], rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
# Corrélation religion → élevage porcin (test logique)
if 'share_muslim_pct' in df.columns and 'target_pig_carcass' in df.columns:
    sub = df.dropna(subset=['share_muslim_pct','target_pig_carcass'])
    r = sub[['share_muslim_pct','target_pig_carcass']].corr().iloc[0,1]
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.scatter(sub['share_muslim_pct'], sub['target_pig_carcass'],
               alpha=0.4, s=15, c=sub['cluster'], cmap='tab10')
    ax.set_xlabel('% Population musulmane (Pew 2010)')
    ax.set_ylabel('target_pig_carcass (kg)')
    ax.set_title(f'Religion musulmane → Poids carcasse porc\nr = {r:+.3f}',
                  weight='bold', fontsize=12)
    plt.tight_layout()
    plt.show()
    print(f'\n→ Pays musulmans : moins de cochons (négatif logique).')

## 7. 🆕 Meat consumption per type (OWID)

**Justification** : 5 colonnes meat (beef, pig, poultry, sheep/goat, other) + lait + œufs → contexte de demande pour les modèles d'élevage.

In [ ]:
meat_cols = [c for c in df.columns if c.startswith('meat_') and c.endswith('_kg_pc')]
if meat_cols:
    # Comparaison consommation par cluster
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Boxplots par cluster
    ax = axes[0]
    melted = df[['cluster'] + meat_cols].melt(id_vars='cluster', var_name='meat_type', value_name='kg_pc')
    melted['meat_type'] = melted['meat_type'].str.replace('meat_','').str.replace('_kg_pc','')
    sns.boxplot(data=melted, x='meat_type', y='kg_pc', hue='cluster', ax=ax, palette='tab10')
    ax.set_title('Consommation viande (kg/hab/an) par type × cluster', weight='bold')
    ax.legend(title='Cluster', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    ax.set_xlabel(''); ax.set_ylabel('kg / habitant / an')

    # Moyennes par cluster
    ax = axes[1]
    cluster_meat = df.groupby('cluster')[meat_cols].mean()
    cluster_meat.columns = [c.replace('meat_','').replace('_kg_pc','') for c in cluster_meat.columns]
    cluster_meat.plot(kind='bar', stacked=True, ax=ax, colormap='Paired', alpha=0.9)
    ax.set_title('Consommation moyenne par cluster (stacked)', weight='bold')
    ax.set_xlabel('Cluster'); ax.set_ylabel('Total kg/hab/an')
    ax.set_xticklabels([f'C{c}' for c in cluster_meat.index], rotation=0)
    ax.legend(loc='upper right', fontsize=8)

    plt.tight_layout()
    plt.show()

## 8. 🆕 Indices de stress climatique (V12)

**Justification** : Heat stress, frost risk, aridity, PET — features dérivées qui capturent les stress que les cultures subissent.

In [ ]:
stress_cols = ['heat_stress_index','frost_risk_index','aridity_index_calc','growing_season_index_v2','continentality']
stress_cols = [c for c in stress_cols if c in df.columns]

if stress_cols:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    for ax, col in zip(axes.flatten(), stress_cols):
        # Moyenne par cluster
        cluster_means = df.groupby('cluster')[col].mean().sort_values()
        ax.bar(cluster_means.index, cluster_means.values,
               color=[plt.cm.tab10(c) for c in cluster_means.index], alpha=0.85)
        ax.set_title(col, weight='bold', fontsize=11)
        ax.set_xlabel('Cluster')

    # Carte mondiale des PET annuels (scatter lat/lon)
    if 'pet_annual' in df.columns:
        ax = axes.flatten()[5]
        recent = df[df['Annee']==2020].dropna(subset=['pet_annual','latitude','longitude'])
        sc = ax.scatter(recent['longitude'], recent['latitude'],
                        c=recent['pet_annual'], cmap='YlOrRd', s=30, alpha=0.8)
        ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
        ax.set_title('PET annuel (2020) — Évapotranspiration', weight='bold')
        plt.colorbar(sc, ax=ax, label='PET (mm/an)')

    plt.suptitle('Indices de stress climatique V12 par cluster', weight='bold', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

## 9. Corrélations features → cibles (heatmap par groupe)

In [ ]:
# 5 cibles fortes + 5 cibles moyennes
KEY_TARGETS = ['target_co2_emissions','target_methane_emissions','target_forest_share',
               'target_thermal_anomaly','target_yield_cereals','target_yield_tomato',
               'target_milk_yield','target_cattle_carcass','target_stunting','target_birth_rate']
KEY_TARGETS = [t for t in KEY_TARGETS if t in df.columns]

GROUPS = {
    'Climat dynamique': [c for c in df.columns if any(k in c for k in ['T_annual','P_annual','T_anomaly','nasa_t2m','nasa_prec','be_t_anom_annual']) and '_lag' not in c and '_roll' not in c],
    'Suitability EcoCrop': [c for c in df.columns if c.startswith('suit_')][:12],
    'Stress climatique': [c for c in df.columns if c in ('heat_stress_index','frost_risk_index','aridity_index_calc','growing_season_index_v2','continentality','pet_annual')],
    'Religion (Pew)': [c for c in df.columns if c.startswith('share_') and ('muslim' in c or 'christian' in c or 'hindu' in c)],
    'Meat per type': [c for c in df.columns if c.startswith('meat_') and c.endswith('_kg_pc')],
    'Sol & humidité': [c for c in df.columns if any(k in c for k in ['clay','sand','soil_pH','organic_carbon','gwetroot','gwettop'])][:8],
}

fig, axes = plt.subplots(len(GROUPS), 1, figsize=(13, 3*len(GROUPS)))
for ax, (gname, gfeats) in zip(axes, GROUPS.items()):
    cols = [c for c in gfeats if c in df.columns]
    if not cols: continue
    cmat = df[cols + KEY_TARGETS].corr().loc[cols, KEY_TARGETS]
    sns.heatmap(cmat, ax=ax, cmap='RdBu_r', center=0, vmin=-0.8, vmax=0.8,
                annot=True, fmt='.2f', cbar=True, annot_kws={'size': 8})
    ax.set_title(f'{gname} → cibles', weight='bold')
    ax.set_xticklabels([t.replace('target_','') for t in KEY_TARGETS], rotation=30, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

## 10. Évolution temporelle (8 pays × 4 cibles)

In [ ]:
PAYS_SELECT = ['FR', 'US', 'CN', 'IN', 'BR', 'NG', 'RU', 'JP']
TIME_TARGETS = ['target_yield_tomato', 'target_thermal_anomaly',
                'target_co2_emissions', 'target_milk_yield']
TIME_TARGETS = [t for t in TIME_TARGETS if t in df.columns]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, tgt in zip(axes.flatten(), TIME_TARGETS):
    for iso in PAYS_SELECT:
        sub = df[df['ISO']==iso].dropna(subset=[tgt]).sort_values('Annee')
        if len(sub) >= 5:
            cluster = sub['cluster'].iloc[0] if 'cluster' in sub.columns else 0
            ax.plot(sub['Annee'], sub[tgt], marker='o', ms=3, lw=1.5,
                    label=f'{iso} (C{cluster})', alpha=0.8)
    ax.set_title(tgt.replace('target_',''), weight='bold')
    ax.set_xlabel('Année'); ax.legend(ncol=4, fontsize=8)
plt.tight_layout()
plt.show()

## 11. Plafond de prédictibilité — comparaison nouvelles features

In [ ]:
def stat_max_corr(tgt, only_groups=None):
    sub = df.dropna(subset=[tgt])
    num = sub.select_dtypes(include='number').columns.tolist()
    excl = set(targets) | {'ISO','Annee','T_ref','P_ref','cluster'}
    feats = [c for c in num if c not in excl]
    if only_groups:
        feats = [c for c in feats if any(p in c for p in only_groups)]
    if not feats: return np.nan
    cors = sub[feats].corrwith(sub[tgt]).abs().dropna()
    return cors.max() if len(cors) else np.nan

# Pour quelques cibles, calculer le max |r| pour chaque groupe de features
TEST_TARGETS = ['target_yield_tomato','target_yield_apple','target_yield_chicken_carcass'.replace('chicken_carcass','potato'),
                'target_milk_yield','target_co2_emissions','target_pig_carcass']
TEST_TARGETS = [t for t in TEST_TARGETS if t in df.columns]

groups_to_test = {
    'Climat de base': ['T_annual','P_annual','temp_mean','precip_mean'],
    'NASA POWER': ['nasa_t2m','nasa_prec','nasa_gwet'],
    'EcoCrop suit': ['suit_'],
    'Stress climat V12': ['heat_stress','frost_risk','aridity','growing_season'],
    'Religion': ['share_'],
    'Meat per type': ['meat_'],
    'Sol': ['clay','sand','soil_pH','organic_carbon'],
}

rows = []
for tgt in TEST_TARGETS:
    if tgt not in df.columns: continue
    row = {'cible': tgt.replace('target_','')}
    for gname, prefixes in groups_to_test.items():
        row[gname] = round(stat_max_corr(tgt, only_groups=prefixes), 3)
    rows.append(row)

if rows:
    summary = pd.DataFrame(rows).set_index('cible')
    print('Max |corrélation| par groupe de features (par cible) :\n')
    print(summary.to_string())

    fig, ax = plt.subplots(figsize=(13, 6))
    sns.heatmap(summary, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1,
                cbar_kws={'label': 'Max |r|'}, ax=ax)
    ax.set_title('Plafond de corrélation par groupe de features × cible',
                  weight='bold', fontsize=13)
    plt.tight_layout()
    plt.show()

## 12. Synthèse — justifications V12

```
NOUVEAUTÉS V12 INTÉGRÉES                          IMPACT
─────────────────────────────────────────────────────────────
🆕 Cultures spécifiques (36)    yield_X agg → spé      ★★★
🆕 EcoCrop suitability (39)     T_opt × P_opt × frost  ★★
🆕 Stress climat (5)            heat/frost/aridity/PET ★★
🆕 Religion Pew (7)             contexte alimentaire   ★
🆕 Meat per type (5)            beef/pig/poultry/...   ★
🆕 Élevage FAO (8)              lait/carcasses/œufs    ★★★
🆕 Émissions atmo (3)           CO2/CH4/N2O            ★★★
🆕 Énergie production (6)       solar/wind/hydro/...   ★★★
```

### Cibles débloquées par V12
- **Émissions CH4/N2O** → R²=0.96 ⭐⭐⭐
- **Production œufs** → R²=0.85 ⭐⭐
- **Rendement lait** → R²=0.83 ⭐⭐
- **Production énergie fossile** → R²=0.85-0.90 ⭐⭐
- **Cultures spécifiques top** : Concombre 0.59, Colza 0.58, Tomate 0.53, Pomme de terre 0.43

### Limites résiduelles
- Carcasses animales : R² 0.2-0.3 (race + intensification non publiques)
- Cultures niche : Mangue, Pois chiche, Cerise toujours R² < 0
- Migration nette : appartient à Couche 4

→ Voir `couche1_planete/modelisation_couche1.ipynb` pour le détail modèles